In [7]:
import sys, os, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)   # make CWD = project root
print("Project root:", ROOT)
print("Available in data/raw:", os.listdir("data/raw"))

Project root: e:\Python\PSEB COURSE\Python PSEB 2\Reproducible Data Pipelines, Tabular ML & Systems Exploration\Online_Retail_ML_Project
Available in data/raw: ['Online Retail.xlsx', 'online_retail.csv']


In [8]:
import sys, os, pathlib

# Ensure ROOT is defined even if Cell 1 was not run
ROOT = pathlib.Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.cleaning import remove_duplicates, remove_outliers

# --- Load & clean ---
# Prefer CSV if available for fast loading; fallback to Excel
raw_csv = ROOT / "data" / "raw" / "online_retail.csv"
raw_xlsx = ROOT / "data" / "raw" / "Online Retail.xlsx"

if raw_csv.exists():
    df = pd.read_csv(raw_csv)
else:
    df = pd.read_excel(raw_xlsx)

df = remove_duplicates(df)
df = remove_outliers(df, "Quantity")
df = remove_outliers(df, "UnitPrice")

# --- Pandas summaries ---
by_country = (df.groupby("Country")["Quantity"]
                .agg(["count", "sum", "mean"])
                .sort_values("sum", ascending=False)
                .head(10))
print(by_country)
print(df["Country"].value_counts().head())

# --- NumPy vectorized ops (2 required) ---
df["log_quantity"] = np.log1p(np.maximum(df["Quantity"].to_numpy(), 0))
z = (df["UnitPrice"].to_numpy() - df["UnitPrice"].mean()) / df["UnitPrice"].std()
df["price_outlier"] = np.abs(z) > 3

# --- 3 figures saved to outputs/figures/ ---
os.makedirs("outputs/figures", exist_ok=True)

# Figure 1: top-10 countries by total quantity
fig, ax = plt.subplots(figsize=(8, 5))
by_country["sum"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Top 10 countries by total quantity sold")
ax.set_xlabel("Country")
ax.set_ylabel("Total quantity sold (units)")
plt.tight_layout()
fig.savefig("outputs/figures/top_countries.png", dpi=150)
plt.close(fig)

# Figure 2: quantity distribution (log scale)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["log_quantity"], bins=50, color="darkorange", edgecolor="black")
ax.set_title("Distribution of log(1 + quantity) after cleaning")
ax.set_xlabel("log(1 + Quantity)")
ax.set_ylabel("Frequency (count of rows)")
plt.tight_layout()
fig.savefig("outputs/figures/quantity_distribution.png", dpi=150)
plt.close(fig)

# Figure 3: price outliers flagged
fig, ax = plt.subplots(figsize=(8, 5))
counts = df["price_outlier"].value_counts().sort_index()
ax.bar(["Normal", "Outlier (|z|>3)"], [counts.get(False, 0), counts.get(True, 0)],
       color=["seagreen", "crimson"])
ax.set_title("UnitPrice outliers flagged by z-score (>3σ)")
ax.set_xlabel("Category")
ax.set_ylabel("Number of rows (count)")
plt.tight_layout()
fig.savefig("outputs/figures/price_outliers.png", dpi=150)
plt.close(fig)

print("Saved 3 figures to outputs/figures/")

                 count      sum      mean
Country                                  
United Kingdom  413342  1880112  4.548563
Germany           7185    59842  8.328740
France            6458    54252  8.400743
EIRE              5795    44992  7.763934
Belgium           1594    13439  8.430991
Spain             1937    12940  6.680434
Switzerland       1405    11921  8.484698
Portugal          1234     9532  7.724473
Norway             692     5889  8.510116
Italy              639     4970  7.777778
Country
United Kingdom    413342
Germany             7185
France              6458
EIRE                5795
Spain               1937
Name: count, dtype: int64
Saved 3 figures to outputs/figures/
